In [1]:
!pip install textattack[tensorflow]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 19.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 42.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2025-04-29 13:51:19--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-04-29 13:51:20--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-04-29 13:51:20--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2
import tensorflow_datasets as tfds
import numpy as np


from tensorflow.keras.models import load_model
import pickle
# Load the model
model = load_model('NLPModel.h5')


# Load the saved tokenizer from the pickle file
with open("tokenizer.pickle", "rb") as handle:
   tokenizer = pickle.load(handle)

# Load Yelp Polarity dataset
dataset, info = tfds.load('yelp_polarity_reviews', with_info=True, as_supervised=True)
train_data, test_data = dataset['train'], dataset['test']

# Prepare the data
vocab_size = 50000  # Limit vocabulary to top 50,000 words
max_length = 200    # Maximum review length (truncation/padding)

# Tokenizer to convert text to sequences
#tokenizer = keras.preprocessing.text.Tokenizer(num_words=vocab_size)
#tokenizer.fit_on_texts([text.numpy().decode('utf-8') for text, _ in train_data])

# Convert text data to sequences and pad them
def preprocess_dataset(dataset):
    texts, labels = [], []
    for text, label in dataset:
        texts.append(text.numpy().decode('utf-8'))
        labels.append(label.numpy())
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    return padded_sequences, np.array(labels)

# Preprocess training and testing data
tr_x, tr_y = preprocess_dataset(train_data)
te_x, te_y = preprocess_dataset(test_data)



Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.L1HGTF_0.2.0/yelp_polarity_reviews-train.…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.L1HGTF_0.2.0/yelp_polarity_reviews-test.t…

Dataset yelp_polarity_reviews downloaded and prepared to /root/tensorflow_datasets/yelp_polarity_reviews/0.2.0. Subsequent calls will reuse this data.


In [4]:
# Load GloVe 6B embeddings
glove_path = "/content/glove.6B.100d.txt"  # Update with the correct path
embedding_dim = 100  # Choose dimension (50d, 100d, 200d, or 300d)

# Create a dictionary mapping words to vectors
embeddings_index = {}
with open(glove_path, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

# Create an embedding matrix for the tokenizer’s vocabulary
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector



In [5]:
def predict_sentiment(model, text, tokenizer, max_length=200):
    # Convert text to sequence using the tokenizer
    sequence = tokenizer.texts_to_sequences([text])

    # Pad the sequence to the fixed length
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')

    # Get prediction probability
    prediction = model.predict(padded_sequence)[0][0]

    # Classify sentiment
    sentiment = "Positive" if prediction > 0.5 else "Negative"

    # Confidence score
    confidence = prediction if sentiment == "Positive" else 1 - prediction

    return sentiment, confidence


In [10]:
import numpy as np
import tensorflow as tf
from textattack import Attack
from textattack.constraints.pre_transformation import (
    RepeatModification,
    StopwordModification,
)
from textattack.constraints.semantics.sentence_encoders import UniversalSentenceEncoder
from textattack.goal_functions import UntargetedClassification
from textattack.search_methods import GreedyWordSwapWIR
from textattack.transformations import (
    CompositeTransformation,
    WordSwapEmbedding,
    WordSwapHomoglyphSwap,
    WordSwapNeighboringCharacterSwap,
    WordSwapRandomCharacterDeletion,
    WordSwapRandomCharacterInsertion,
)

In [11]:
from textattack.models.wrappers import ModelWrapper
import numpy as np
class CustomKerasModelWrapper(ModelWrapper):
    def __init__(self, model, tokenizer, max_length=200):
        self.model = model
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, text_input_list):
        sequences = self.tokenizer.texts_to_sequences(text_input_list)
        padded = pad_sequences(sequences, maxlen=self.max_length, padding='post', truncating='post')

        # Get raw probabilities (shape [batch_size, 1])
        preds = self.model.predict(padded)

        # Convert to logits for both classes (shape [batch_size, 2])
        # For binary classification: class 0 = 1 - prob, class 1 = prob
        return np.concatenate([1 - preds, preds], axis=1)


model_wrapper = CustomKerasModelWrapper(model, tokenizer, max_length=max_length)


In [22]:
from textattack.constraints.overlap import MaxWordsPerturbed
transformation = CompositeTransformation(
            [
                # (1) Insert: Insert a space into the word.
                # Generally, words are segmented by spaces in English. Therefore,
                # we can deceive classifiers by inserting spaces into words.
                WordSwapRandomCharacterInsertion(
                    random_one=True,
                    letters_to_insert=" ",
                    skip_first_char=True,
                    skip_last_char=True,
                ),
                # (2) Delete: Delete a random character of the word except for the first
                # and the last character.
                WordSwapRandomCharacterDeletion(
                    random_one=True, skip_first_char=True, skip_last_char=True
                ),
                # (3) Swap: Swap random two adjacent letters in the word but do not
                # alter the first or last letter. This is a common occurrence when
                # typing quickly and is easy to implement.
                WordSwapNeighboringCharacterSwap(
                    random_one=True, skip_first_char=True, skip_last_char=True
                ),
                # (4) Substitute-C (Sub-C): Replace characters with visually similar
                # characters (e.g., replacing “o” with “0”, “l” with “1”, “a” with “@”)
                # or adjacent characters in the keyboard (e.g., replacing “m” with “n”).
                WordSwapHomoglyphSwap(),
                # (5) Substitute-W
                # (Sub-W): Replace a word with its topk nearest neighbors in a
                # context-aware word vector space. Specifically, we use the pre-trained
                # GloVe model [30] provided by Stanford for word embedding and set
                # topk = 5 in the experiment.
                WordSwapEmbedding(max_candidates=5),
            ]
)
constraints = [
    RepeatModification(),
    StopwordModification(),
    UniversalSentenceEncoder(threshold=0.8),  # Paper specifies ≤2 modifications
]

search_method = GreedyWordSwapWIR(wir_method="delete")
attack = Attack(
    UntargetedClassification(model_wrapper),
    constraints,
    transformation,
    GreedyWordSwapWIR(wir_method="delete")
)

textattack: Unknown if model of class <class 'keras.src.models.sequential.Sequential'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.


In [17]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')
from textattack.attack_results import (
    AttackResult,
    SuccessfulAttackResult,
    FailedAttackResult,
    SkippedAttackResult
)
input_text = "This restaurant was absolutely amazing! The food was delicious and the service was great."
true_label = 1  # Must match your model's label index

from textattack.attack_results import SuccessfulAttackResult, FailedAttackResult

attack_result = attack.attack(input_text, true_label)

if isinstance(attack_result, SuccessfulAttackResult):
    print(f"""
    Original Text: {attack_result.original_text()}
    Modified Text: {attack_result.perturbed_text()}  # No parentheses needed
    Original Prediction: {attack_result.original_result.output}
    New Prediction: {attack_result.perturbed_result.output}
    """)
else:
    print("Attack failed")


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 621ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 553ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step

    Original Text: This restaurant was absolutely amazing! The food was delicious and the service was great.
    Modified Text: This restaurant was absolutely astonishing! The food was delicious and the service was grat.  # No parentheses needed
    Original Prediction: 1
    New Prediction: 0
    


In [23]:
import numpy as np

# Take first 100 preprocessed samples
sample_size = 100
te_x_sample = te_x[:sample_size]
te_y_sample = te_y[:sample_size]

# Reconstruct raw texts from tokenizer (approximate)
reverse_word_map = dict(map(reversed, tokenizer.word_index.items()))
test_texts = [
    " ".join([reverse_word_map.get(idx, "") for idx in seq if idx != 0])
    for seq in te_x_sample
]
# Original predictions
baseline_preds = model.predict(te_x_sample)
baseline_accuracy = np.mean(np.round(baseline_preds).flatten() == te_y_sample)
print(f"Baseline Accuracy: {baseline_accuracy:.2%}")


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Baseline Accuracy: 97.00%


In [24]:
from textattack.datasets import Dataset

# Filter to samples the model originally got RIGHT
original_correct_mask = (np.round(baseline_preds).flatten() == te_y_sample)
originally_correct_indices = np.where(original_correct_mask)[0]
num_originally_correct = len(originally_correct_indices)

print(f"Samples originally correct: {num_originally_correct}/{sample_size}")

# Subset to only these correctly-classified samples
attack_texts = [test_texts[i] for i in originally_correct_indices]
attack_labels = te_y_sample[originally_correct_indices]
attack_labels_fixed = [int(label) for label in attack_labels]  # Convert to Python int

# Attack only these samples
from textattack import Attacker, AttackArgs

attack_args = AttackArgs(
    num_examples=num_originally_correct,
    log_to_csv="attack_log.csv",
    disable_stdout=True
)
attacker = Attacker(attack, Dataset(list(zip(attack_texts, attack_labels_fixed))), attack_args)  # Use fixed labels
attack_results = attacker.attack_dataset()


# Calculate metrics per paper
successful_attacks = sum(1 for res in attack_results if isinstance(res, SuccessfulAttackResult))
skipped_attacks = sum(1 for res in attack_results if isinstance(res, SkippedAttackResult))
failed_attacks = num_originally_correct - successful_attacks - skipped_attacks

# Paper-style metrics
original_accuracy = num_originally_correct / sample_size  # Baseline accuracy
attack_success_rate = successful_attacks / (num_originally_correct - skipped_attacks)  # ASR
accuracy_under_attack = (num_originally_correct - successful_attacks) / sample_size  # AuA

print(f"""
+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | {original_accuracy:.2%}  |
| Accuracy Under Attack     | {accuracy_under_attack:.2%}  |
| Attack Success Rate (ASR) | {attack_success_rate:.2%}  |
| Successful Attacks        | {successful_attacks}       |
| Failed Attacks            | {failed_attacks}       |
| Skipped Attacks           | {skipped_attacks}       |
+---------------------------+-----------+
""")


textattack: Logging to CSV at path attack_log.csv


Samples originally correct: 97/100
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  CompositeTransformation(
    (0): WordSwapRandomCharacterInsertion(
        (random_one):  True
      )
    (1): WordSwapRandomCharacterDeletion(
        (random_one):  True
      )
    (2): WordSwapNeighboringCharacterSwap(
        (random_one):  True
      )
    (3): WordSwapHomoglyphSwap
    (4): WordSwapEmbedding(
        (max_candidates):  5
        (embedding):  WordEmbedding
      )
    )
  (constraints): 
    (0): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.8
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 



  0%|          | 0/97 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step


[Succeeded / Failed / Skipped / Total] 0 / 1 / 0 / 1:   1%|          | 1/97 [00:14<23:25, 14.64s/it]

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step


[Succeeded / Failed / Skipped / Total] 87 / 9 / 0 / 96:  99%|█████████▉| 96/97 [03:12<00:02,  2.01s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


[Succeeded / Failed / Skipped / Total] 88 / 9 / 0 / 97: 100%|██████████| 97/97 [03:14<00:00,  2.01s/it]


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 88     |
| Number of failed attacks:     | 9      |
| Number of skipped attacks:    | 0      |
| Original accuracy:            | 100.0% |
| Accuracy under attack:        | 9.28%  |
| Attack success rate:          | 90.72% |
| Average perturbed word %:     | 38.17% |
| Average num. words per input: | 101.76 |
| Avg num queries:              | 139.24 |
+-------------------------------+--------+

+---------------------------+-----------+
| Metric                    | Value     |
+---------------------------+-----------+
| Original Accuracy         | 97.00%  |
| Accuracy Under Attack     | 9.00%  |
| Attack Success Rate (ASR) | 90.72%  |
| Successful Attacks        | 88       |
| Failed Attacks            | 9       |
| Skipped Attacks           | 0       |
+---------------------------+-----------+



In [20]:
# New: Print example attacks ==================================================
num_examples_to_show = 3  # How many examples to display
successful_results = [r for r in attack_results if isinstance(r, SuccessfulAttackResult)]

if successful_results:
    print("\n=== Adversarial Examples ===")
    for i, result in enumerate(successful_results[:num_examples_to_show]):
        original_text = result.original_result.attacked_text.text
        perturbed_text = result.perturbed_result.attacked_text.text

        print(f"\nExample {i+1}:")
        print(f"Original: {original_text}")
        print(f"Perturbed: {perturbed_text}")
        print(f"Original prediction: {result.original_result.output}")
        print(f"Perturbed prediction: {result.perturbed_result.output}")
        print("-" * 50)


=== Adversarial Examples ===

Example 1:
Original: i went in to purchase and was treated so rudely i had to walk out even though he had the item i wanted i will never step foot in this of business again
Perturbed: i went in to purchase and was treated so shamefully i had to walk out even though he had the item i wanted i will n ever step foot in this of business again
Original prediction: 0
Perturbed prediction: 1
--------------------------------------------------

Example 2:
Original: this place really is horrible every time i wind up getting convinced to go here i always walk out feeling like my pocket has been picked the food isn't bad but at roughly 25 30 per person there are so many better places to eat to hell with olive garden n nand this particular one has the worst smelling parking lot in las vegas every time i drive by to go to b j 's nearby which is a thousand times better it smells like a sewer main broke or something n n2 stars only because the wait staff is friendly
Pert